# BNET Complex RF Full-Vector Test

Self-contained test for the dual-cable RF capture path:

- RF OUT1 -> RF IN1 captures even BNET samples.
- RF OUT2 -> RF IN2 captures odd BNET samples.
- The notebook reconstructs the full BNET vector and compares it with a PC reference.

The test intentionally uses a mixed input waveform and non-identity staged weights so the result is more complicated than the ramp smoke test.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
SCPI_CLIENT_PATH = ROOT / "Examples" / "python"
if str(SCPI_CLIENT_PATH) not in sys.path:
    sys.path.insert(0, str(SCPI_CLIENT_PATH))

import redpitaya_scpi as scpi

RP_IP = "169.254.77.151"
RP_PORT = 5000
TIMEOUT_S = 10

BNET_VECTOR_LEN = 2048
BNET_STAGE_COUNT = int(np.log2(BNET_VECTOR_LEN))
BNET_TOTAL_WEIGHTS = BNET_VECTOR_LEN * BNET_STAGE_COUNT
BNET_WEIGHT_FRAC = 6

print(f"Complex BNET RF notebook ready for {RP_IP}:{RP_PORT}")

In [ ]:
rp = scpi.scpi(RP_IP, timeout=TIMEOUT_S, port=RP_PORT)

def cmd(message: str, check: bool = True) -> None:
    print(f">> {message}")
    rp.tx_txt(message)
    if check:
        rp.check_error(stop=False)

def query(message: str, check: bool = True) -> str:
    print(f">> {message}")
    rp.tx_txt(message)
    response = rp.rx_txt().strip()
    print(f"<< {response}")
    if check:
        rp.check_error(stop=False)
    return response

def query_int(message: str) -> int:
    response = query(message)
    return int(response, 0)

def next_scpi_error() -> str:
    print(">> SYST:ERR:NEXT?")
    rp.tx_txt("SYST:ERR:NEXT?")
    err = rp.rx_txt().strip()
    print(f"<< {err}")
    return err

def assert_no_scpi_error(context: str) -> None:
    err = next_scpi_error()
    if not err.startswith("0,"):
        raise RuntimeError(f"{context} failed with SCPI error: {err}")

def strict_cmd(message: str) -> None:
    print(f">> {message}")
    rp.tx_txt(message)
    assert_no_scpi_error(message)

def clear_errors() -> None:
    for _ in range(16):
        if next_scpi_error().startswith("0,"):
            return

clear_errors()

In [ ]:
def align_up(value: int, alignment: int) -> int:
    return ((value + alignment - 1) // alignment) * alignment

def quantize_sample_i14(values: np.ndarray) -> np.ndarray:
    scaled = np.rint(np.asarray(values, dtype=np.float64) * ((1 << 13) - 1))
    clipped = np.clip(scaled, -(1 << 13), (1 << 13) - 1)
    return clipped.astype("<i2")

def quantize_weight_q6(values: np.ndarray) -> np.ndarray:
    scaled = np.rint(np.asarray(values, dtype=np.float64) * (1 << BNET_WEIGHT_FRAC))
    clipped = np.clip(scaled, -(1 << 6), (1 << 6) - 1)
    return clipped.astype(np.int16)

def pack_weight_word(y0_weight: int, y1_weight: int) -> np.int16:
    high = int(y0_weight) & 0x7f
    low = int(y1_weight) & 0x7f
    packed_u14 = (high << 7) | low
    return np.int16(packed_u14 if packed_u14 < 0x2000 else packed_u14 - 0x4000)

def unpack_s7(value: int) -> int:
    value &= 0x7f
    return value - 0x80 if value & 0x40 else value

def round_shift_q6(value: int) -> int:
    bias = 1 << (BNET_WEIGHT_FRAC - 1)
    return (value - bias if value < 0 else value + bias) >> BNET_WEIGHT_FRAC

def saturate_i14(value: int) -> int:
    return max(-(1 << 13), min((1 << 13) - 1, int(value)))

def butterfly_reference_i14(samples_i14: np.ndarray, packed_weights: np.ndarray) -> np.ndarray:
    vector = np.asarray(samples_i14, dtype=np.int64).copy()
    vector_len = len(vector)
    stage_count = int(np.log2(vector_len))
    packed = np.asarray(packed_weights, dtype=np.int16).astype(np.int64)
    assert packed.size == vector_len * stage_count
    for stage in range(stage_count):
        next_vector = np.zeros_like(vector)
        half_span = 1 << stage
        for pair_idx in range(vector_len // 2):
            pair_in_group = pair_idx & (half_span - 1)
            group_base = (pair_idx - pair_in_group) << 1
            addr_a = group_base + pair_in_group
            addr_b = addr_a + half_span
            a = int(vector[addr_a])
            b = int(vector[addr_b])
            wa = int(packed[stage * vector_len + addr_a]) & 0x3fff
            wb = int(packed[stage * vector_len + addr_b]) & 0x3fff
            wa_y0 = unpack_s7(wa >> 7)
            wa_y1 = unpack_s7(wa)
            wb_y0 = unpack_s7(wb >> 7)
            wb_y1 = unpack_s7(wb)
            next_vector[addr_a] = saturate_i14(round_shift_q6(a * wa_y0 + b * wb_y0))
            next_vector[addr_b] = saturate_i14(round_shift_q6(a * wa_y1 + b * wb_y1))
        vector = next_vector
    return vector.astype("<i2")

def pack_i16_le(values: np.ndarray) -> bytes:
    return np.asarray(values, dtype="<i2").tobytes()

def bytes_to_scpi_csv(data: bytes) -> str:
    return ",".join(str(b) for b in data)

In [ ]:
def reserve_bnet_slot(slot: int, address: int, size_bytes: int) -> None:
    strict_cmd(f"BNET:DDR:SLOT{slot}:RESERVE {address},{size_bytes}")

def upload_bnet_slot_bytes(slot: int, offset_bytes: int, data: bytes) -> None:
    for chunk_offset in range(0, len(data), 1024):
        chunk = data[chunk_offset:chunk_offset + 1024]
        payload = bytes_to_scpi_csv(chunk)
        strict_cmd(f"BNET:DDR:SLOT{slot}:OFFSET{offset_bytes + chunk_offset}:DATA{len(chunk)} {payload}")

def attach_slot_to_stream(slot: int, stream: int, buffer: int) -> None:
    strict_cmd(f"BNET:DDR:SLOT{slot}:STREAM{stream}:BUF{buffer}")

def spaced_slot_address(ddr_start: int, slot_index: int, slot_size_bytes: int) -> int:
    # Keep slot mappings page-separated. Earlier tests showed adjacent mappings can fail.
    stride = align_up(slot_size_bytes, 0x1000) + 0x1000
    return align_up(ddr_start + slot_index * stride, 0x1000)

def bnet_diagnostic_snapshot(label: str) -> dict[str, int]:
    print(f"\n{label}")
    snapshot = {
        "status": query_int("BNET:STATUS?"),
        "active": query_int("BNET:ACTIVE?"),
        "pending": query_int("BNET:PENDING?"),
        "error": query_int("BNET:ERROR?"),
        "stream0_rptr": query_int("BNET:STREAM0:RPTR?"),
        "stream1_rptr": query_int("BNET:STREAM1:RPTR?"),
        "time_total": query_int("BNET:TIME0?"),
        "time_load": query_int("BNET:TIME1?"),
        "time_compute": query_int("BNET:TIME2?"),
        "time_playback": query_int("BNET:TIME3?"),
    }
    for key, value in snapshot.items():
        print(f"  {key:14s} = {value if 'status' not in key and key != 'error' else hex(value)}")
    return snapshot

In [ ]:
def make_complex_bnet_input(vector_len: int = BNET_VECTOR_LEN) -> np.ndarray:
    x = np.arange(vector_len, dtype=np.float64)
    y = (
        0.18 * np.sin(2 * np.pi * 7 * x / vector_len + 0.2)
        + 0.10 * np.sin(2 * np.pi * 31 * x / vector_len + 1.1)
        + 0.07 * np.cos(2 * np.pi * 73 * x / vector_len)
        + 0.04 * np.sign(np.sin(2 * np.pi * 5 * x / vector_len))
    )
    envelope = 0.85 + 0.15 * np.sin(2 * np.pi * 3 * x / vector_len)
    return quantize_sample_i14(y * envelope)

def make_alternating_mix_stage_weights(vector_len: int = BNET_VECTOR_LEN) -> np.ndarray:
    stage_count = int(np.log2(vector_len))
    q = quantize_weight_q6(np.array([0.78, 0.42, -0.42, 0.78, 0.62, -0.55, 0.55, 0.62], dtype=np.float64))
    a0, b0, a1, b1, c0, d0, c1, d1 = [int(v) for v in q]
    weights = np.zeros(stage_count * vector_len, dtype="<i2")
    for stage in range(stage_count):
        half_span = 1 << stage
        if stage % 3 == 0:
            wa_y0, wa_y1, wb_y0, wb_y1 = a0, a1, b0, b1
        elif stage % 3 == 1:
            wa_y0, wa_y1, wb_y0, wb_y1 = c0, c1, d0, d1
        else:
            wa_y0, wa_y1, wb_y0, wb_y1 = 63, 0, 0, 63
        for pair_idx in range(vector_len // 2):
            pair_in_group = pair_idx & (half_span - 1)
            group_base = (pair_idx - pair_in_group) << 1
            addr_a = group_base + pair_in_group
            addr_b = addr_a + half_span
            base = stage * vector_len
            weights[base + addr_a] = pack_weight_word(wa_y0, wa_y1)
            weights[base + addr_b] = pack_weight_word(wb_y0, wb_y1)
    return weights

In [ ]:
def run_bnet_ddr_payload(input_samples: np.ndarray, weights: np.ndarray, vector_len: int = BNET_VECTOR_LEN) -> dict[str, object]:
    clear_errors()
    input_samples = np.asarray(input_samples, dtype="<i2")
    weights = np.asarray(weights, dtype="<i2")
    stage_count = int(np.log2(vector_len))
    assert len(input_samples) == vector_len
    assert len(weights) == vector_len * stage_count

    input_bytes = vector_len * 2
    weight_bytes = vector_len * stage_count * 2
    slot0_size_bytes = align_up(input_bytes, 0x80)
    slot1_size_bytes = align_up(weight_bytes, 0x80)
    expected = butterfly_reference_i14(input_samples, weights)

    ddr_start = query_int("BNET:DDR:START?")
    ddr_size = query_int("BNET:DDR:SIZE?")
    slot0_addr = spaced_slot_address(ddr_start, 0, slot0_size_bytes)
    slot1_addr = spaced_slot_address(ddr_start, 1, slot0_size_bytes)
    assert slot1_addr + slot1_size_bytes <= ddr_start + ddr_size
    print(f"Slot 0 address: 0x{slot0_addr:08x}")
    print(f"Slot 1 address: 0x{slot1_addr:08x}")
    print(f"Payload sizes: stream0={input_bytes} bytes, stream1={weight_bytes} bytes")
    print(f"Expected final range: {expected.min()}..{expected.max()} i14 counts")

    cmd("BNET:RST")
    reserve_bnet_slot(0, slot0_addr, slot0_size_bytes)
    reserve_bnet_slot(1, slot1_addr, slot1_size_bytes)
    upload_bnet_slot_bytes(0, 0, pack_i16_le(input_samples))
    upload_bnet_slot_bytes(1, 0, pack_i16_le(weights))
    attach_slot_to_stream(0, stream=0, buffer=0)
    attach_slot_to_stream(1, stream=1, buffer=0)

    strict_cmd(f"BNET:VLEN {vector_len}")
    strict_cmd("BNET:STREAM0:FORMAT 0")
    strict_cmd("BNET:STREAM1:FORMAT 0")
    strict_cmd("BNET:STREAM0:ENABLE 1")
    strict_cmd("BNET:STREAM1:ENABLE 1")
    strict_cmd("BNET:STREAM0:COMMIT0")
    strict_cmd("BNET:STREAM1:COMMIT0")
    strict_cmd("BNET:MODE DDR")
    assert query("BNET:MODE?") == "DDR"

    before = bnet_diagnostic_snapshot("BNET before START")
    strict_cmd("BNET:START")
    deadline = time.time() + 10.0
    status = 0
    while time.time() < deadline:
        status = query_int("BNET:STATUS?")
        if status & 0x2:
            break
        time.sleep(0.02)
    else:
        bnet_diagnostic_snapshot("BNET timeout")
        raise TimeoutError(f"BNET did not report done; last status=0x{status:x}")

    after = bnet_diagnostic_snapshot("BNET after START")
    assert after["error"] == 0
    assert after["stream0_rptr"] >= input_bytes
    assert after["stream1_rptr"] >= weight_bytes
    return {"input_samples": input_samples, "weights": weights, "expected": expected, "before": before, "after": after}


In [ ]:
def parse_scpi_ascii_array(text: str) -> np.ndarray:
    stripped = text.strip().strip("{}")
    if not stripped:
        return np.array([], dtype=np.float64)
    return np.array([float(item) for item in stripped.split(",") if item.strip()], dtype=np.float64)

def wait_for_trigger(timeout_s: float = 3.0) -> str:
    deadline = time.time() + timeout_s
    last = ""
    while time.time() < deadline:
        last = query("ACQ:TRig:STAT?", check=False)
        if last.upper().startswith("TD"):
            return last
        time.sleep(0.05)
    return last

def acquire_inputs_latest(samples: int = BNET_VECTOR_LEN, trigger_delay_samples: int = 0) -> tuple[np.ndarray, np.ndarray]:
    clear_errors()
    cmd("ACQ:RST")
    cmd("ACQ:DATA:Units VOLTS")
    cmd("ACQ:DATA:FORMAT ASCII")
    cmd("ACQ:DEC 1")
    cmd("ACQ:AVG ON")
    cmd(f"ACQ:TRig:DLY {trigger_delay_samples}")
    cmd("ACQ:TRig:LEV 0")
    cmd("ACQ:START")
    time.sleep(0.1)
    cmd("ACQ:TRig NOW")
    print("Trigger state:", wait_for_trigger())
    captures = []
    for channel in (1, 2):
        rp.tx_txt(f"ACQ:SOUR{channel}:DATA:LATest:N? {samples}")
        data_text = rp.rx_txt()
        rp.check_error(stop=False)
        captures.append(parse_scpi_ascii_array(data_text))
    cmd("ACQ:STOP")
    return captures[0], captures[1]


In [ ]:
def rf_normalize(values: np.ndarray) -> np.ndarray:
    centered = np.asarray(values, dtype=np.float64) - np.mean(values)
    rms = np.sqrt(np.mean(centered ** 2))
    return centered if rms < 1e-12 else centered / rms

def compare_rf_capture_to_pc_reference(captured: np.ndarray, expected_i14: np.ndarray) -> dict[str, float]:
    reference_volts = np.asarray(expected_i14, dtype=np.float64) / ((1 << 13) - 1)
    ref = rf_normalize(reference_volts)
    cap = rf_normalize(captured)
    window = len(ref)
    assert len(cap) >= window
    best = {"correlation": -np.inf, "rmse": np.inf, "offset": 0.0}
    for offset in range(0, len(cap) - window + 1):
        segment = cap[offset:offset + window]
        corr = float(np.mean(segment * ref))
        rmse = float(np.sqrt(np.mean((segment - ref) ** 2)))
        if corr > best["correlation"]:
            best = {"correlation": corr, "rmse": rmse, "offset": float(offset)}
    return best

def extract_one_bnet_cycle_from_dual_capture(cap_in1: np.ndarray, cap_in2: np.ndarray, expected_full_i14: np.ndarray) -> dict[str, object]:
    expected_full_i14 = np.asarray(expected_full_i14, dtype=np.int16)
    expected_out1 = expected_full_i14[0::2]
    expected_out2 = expected_full_i14[1::2]
    half_len = len(expected_out1)
    cmp1 = compare_rf_capture_to_pc_reference(cap_in1, expected_out1)
    cmp2 = compare_rf_capture_to_pc_reference(cap_in2, expected_out2)
    off1 = int(cmp1["offset"])
    off2 = int(cmp2["offset"])
    one_in1 = np.asarray(cap_in1[off1:off1 + half_len], dtype=np.float64)
    one_in2 = np.asarray(cap_in2[off2:off2 + half_len], dtype=np.float64)
    assert len(one_in1) == half_len and len(one_in2) == half_len

    reconstructed_volts = np.empty(len(expected_full_i14), dtype=np.float64)
    reconstructed_volts[0::2] = one_in1
    reconstructed_volts[1::2] = one_in2
    expected_volts = expected_full_i14.astype(np.float64) / ((1 << 13) - 1)

    reconstructed_norm = np.empty_like(reconstructed_volts)
    expected_norm = np.empty_like(expected_volts)
    reconstructed_norm[0::2] = rf_normalize(one_in1)
    reconstructed_norm[1::2] = rf_normalize(one_in2)
    expected_norm[0::2] = rf_normalize(expected_volts[0::2])
    expected_norm[1::2] = rf_normalize(expected_volts[1::2])
    full_corr = float(np.mean(reconstructed_norm * expected_norm))
    full_rmse = float(np.sqrt(np.mean((reconstructed_norm - expected_norm) ** 2)))

    print(f"IN1/OUT1: corr={cmp1['correlation']:.3f}, rmse={cmp1['rmse']:.3f}, offset={off1}")
    print(f"IN2/OUT2: corr={cmp2['correlation']:.3f}, rmse={cmp2['rmse']:.3f}, offset={off2}")
    print(f"Full interleaved normalized result: corr={full_corr:.3f}, rmse={full_rmse:.3f}, channel_skew={off2 - off1}")
    return {
        "in1_cycle": one_in1,
        "in2_cycle": one_in2,
        "reconstructed_volts": reconstructed_volts,
        "reconstructed_norm": reconstructed_norm,
        "expected_norm": expected_norm,
        "cmp_in1": cmp1,
        "cmp_in2": cmp2,
        "full_corr": full_corr,
        "full_rmse": full_rmse,
        "channel_skew": off2 - off1,
    }


In [ ]:
input_samples = make_complex_bnet_input(BNET_VECTOR_LEN)
weights = make_alternating_mix_stage_weights(BNET_VECTOR_LEN)
complex_result = run_bnet_ddr_payload(input_samples, weights)

strict_cmd("OUTPUT1:STATE ON")
strict_cmd("OUTPUT2:STATE ON")

# Capture two half-vector periods so correlation can pick one exact cycle start.
cap_in1, cap_in2 = acquire_inputs_latest(samples=BNET_VECTOR_LEN, trigger_delay_samples=0)
full_bnet_rf = extract_one_bnet_cycle_from_dual_capture(cap_in1, cap_in2, complex_result["expected"])
full_bnet_rf["raw_in1"] = cap_in1
full_bnet_rf["raw_in2"] = cap_in2


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(full_bnet_rf["reconstructed_norm"], label="RF reconstructed full BNET", linewidth=1.1)
plt.plot(full_bnet_rf["expected_norm"], label="PC expected full BNET", linewidth=1.0, alpha=0.8)
plt.title("Complex full BNET result reconstructed from RF IN1/IN2")
plt.xlabel("Full BNET sample index")
plt.ylabel("Normalized amplitude")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(full_bnet_rf["raw_in1"], label="Raw RF IN1 capture", alpha=0.8)
plt.plot(full_bnet_rf["raw_in2"], label="Raw RF IN2 capture", alpha=0.8)
plt.axvline(full_bnet_rf["cmp_in1"]["offset"], color="C0", linestyle="--", label="IN1 selected cycle start")
plt.axvline(full_bnet_rf["cmp_in2"]["offset"], color="C1", linestyle="--", label="IN2 selected cycle start")
plt.title("Raw dual-channel capture and selected cycle starts")
plt.xlabel("Acquisition sample index")
plt.ylabel("Volts")
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
def cleanup():
    try:
        cmd("OUTPUT1:STATE OFF", check=False)
        cmd("OUTPUT2:STATE OFF", check=False)
        cmd("ACQ:STOP", check=False)
    finally:
        rp.close()

# cleanup()